# 08. Umbralización

**Objetivo:** separar píxeles en dos grupos mediante un umbral y comparar estrategias globales y locales.

In [ ]:
import numpy as np
import cv2

from filtrado_digital.io import cargar_imagen, a_grises, ruta_imagen_ejemplo
from filtrado_digital.visualizacion import mostrar_imagen, comparar
from filtrado_digital.sinteticas import agregar_ruido_gaussiano
from filtrado_digital.thresholding import (
    umbral_global_manual,
    otsu_manual,
    otsu_opencv,
    umbral_adaptativo_media_manual,
)

## 1. Segmentación, imagen binaria y umbral

La **segmentación** divide una imagen en regiones o grupos de píxeles. Aquí utilizaremos la intensidad como criterio.

Una **imagen binaria** contiene solo dos valores, por ejemplo `0` y `255`. Un **umbral** `T` es un valor límite:

```text
si intensidad > T  → 255
si intensidad <= T → 0
```

In [ ]:
foto = cargar_imagen(ruta_imagen_ejemplo())
gris = a_grises(foto)
imagen = gris[250:442, 480:672]
imagen = agregar_ruido_gaussiano(imagen, sigma=5)
mostrar_imagen(imagen, "Región de trabajo")

## 2. Umbral global

Un umbral global utiliza el mismo valor `T` en toda la imagen. Funciona mejor cuando los grupos de intensidad están claramente separados.

In [ ]:
binaria_90 = umbral_global_manual(imagen, 90)
binaria_140 = umbral_global_manual(imagen, 140)
comparar([imagen, binaria_90, binaria_140], ["Original", "T=90", "T=140"])

## 3. Método de Otsu

Otsu evalúa posibles umbrales y elige uno que maximiza la separación entre dos clases de intensidades. Así evita seleccionar `T` manualmente.

In [ ]:
t_manual, b_manual = otsu_manual(imagen)
t_cv, b_cv = otsu_opencv(imagen)
print("Umbral manual:", t_manual)
print("Umbral OpenCV:", t_cv)
comparar([b_manual, b_cv], ["Otsu manual", "Otsu OpenCV"])

## 4. Umbral adaptativo

Cuando la iluminación cambia espacialmente, un único umbral puede perder regiones. La umbralización adaptativa calcula un límite local a partir de una vecindad alrededor de cada píxel.

In [ ]:
iluminacion = np.linspace(0.55, 1.15, imagen.shape[1])
no_uniforme = np.clip(imagen * iluminacion[None, :], 0, 255).astype(np.uint8)

adapt_manual = umbral_adaptativo_media_manual(no_uniforme, 15, 4)
adapt_cv = cv2.adaptiveThreshold(
    no_uniforme, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY, 15, 4,
)
comparar([no_uniforme, adapt_manual, adapt_cv], ["No uniforme", "Adaptativo manual", "Adaptativo OpenCV"])

## Conclusiones

- Un umbral global aplica el mismo límite a toda la imagen.
- Otsu selecciona automáticamente un umbral global.
- El umbral adaptativo es útil cuando la iluminación cambia de una región a otra.
- La elección del método depende de la distribución de intensidades y de la iluminación.